# 04 · Adversarial attacks
Paper protocol on the accepted hit trajectories (Black texture). Each run writes snapshots and a loss history to `PGD/`, `CW/`, `NES/`; the last cell aggregates them into R and LERS (bootstrap over trajectories).

In [ ]:
import json, torch, numpy as np
from flynet_robustness import config
from flynet_robustness.models import get_flynet
from flynet_robustness.feature_extraction import load_cnn_network
from flynet_robustness.decoders import load_decoder
from flynet_robustness.metrics import bootstrap_robustness, lers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
network_id = 'FlyNet'   # any registry id: 'CTRNN', 'sCTRNN', 'flynet_degree_matched_0', 'vCNN_5_frames', 'RAFT_12', ...
if config.is_cnn(network_id):
    net = load_cnn_network(network_id, device=device)
else:
    model1, model2, idxs1, idxs2 = get_flynet(network_id, device=device)
    net = (model1, idxs1, model2, idxs2)
decoder = load_decoder(network_id)
trajectories = json.load(open('data/configs/attack_trajectories.json'))   # accepted hit trajectories
from flynet_robustness.attacks import run_pgd, run_cw, run_nes
from flynet_robustness.attacks.common import final_values
attack_kwargs = [dict(config.DEFAULT_STIMULUS, **t, device=device, a=1, mode='avg_pool_adaptive') for t in trajectories]

## PGD — 100 steps, step ε/10, 2 restarts, 15 trajectories

In [ ]:
PGD_EPS = {'l2': [100, 250, 500, 1000, 2000], 'linf': [1, 6, 8, 10, 15]}
for norm, epss in PGD_EPS.items():
    for eps in epss:
        for i, kw in enumerate(attack_kwargs[:15]):
            torch.manual_seed(2 * (i + 1))
            run_pgd(net, decoder, network_id, norm=norm, eps=eps, lr=eps / 10, steps=100, n_restarts=2,
                    base_kwargs=kw, run_id=i + 1, device=device)

## CW — Adam lr 0.1, 150 steps, 10 trajectories  ·  NES — 15 samples/step, 100 steps, lr 1, 15 trajectories

In [ ]:
CW_C, NES_SIGMA = [1, 10, 1e2, 1e3, 1e4, 1e5], [1, 3, 6, 9, 12]
for i, kw in enumerate(attack_kwargs[:10]):
    for c in CW_C:
        for norm in ('linf', 'l2'):
            torch.manual_seed(2 * (i + 1))
            run_cw(net, decoder, network_id, norm=norm, c=c, steps=150, lr=0.1, base_kwargs=kw, run_id=i + 1, device=device)
for i, kw in enumerate(attack_kwargs[:15]):
    for sigma in NES_SIGMA:
        torch.manual_seed(2 * (i + 1))
        run_nes(net, decoder, network_id, sigma=sigma, steps=100, n_samples=15, lr=1.0, base_kwargs=kw,
                run_id=i + 1, device=device)

## Robustness and efficiency
Accuracy = 1 − fraction of initially-correct frames flipped (PGD: best restart). CW: attack success and texture change (L2 norm / L∞) of the successful attacks, per c.

In [ ]:
for name, root, norm, values, n in [('PGD L2', 'PGD', 'l2', PGD_EPS['l2'], 15), ('PGD Linf', 'PGD', 'linf', PGD_EPS['linf'], 15),
                                     ('NES', 'NES', 'gaussian', NES_SIGMA, 15)]:
    acc = 1 - final_values(root, 'Black', network_id, norm, values, n)
    R, R_std = bootstrap_robustness(values, acc)
    print(f'{name:9s} R = {R:.3f} ± {R_std:.3f}   LERS = {lers(R, network_id):.3f} ± {lers(R_std, network_id):.3f}')
for norm in ('linf', 'l2'):
    success = final_values('CW', 'Black', network_id, norm, CW_C, 10)
    size = np.where(success > 0, final_values('CW', 'Black', network_id, norm, CW_C, 10, key='max_delta'), np.nan)
    print(f'CW {norm:4s} success {np.nanmean(success, 1).round(3)}  size {np.nanmean(size, 1).round(3)}')

Counterfactual restoration and the illusion analyses: notebook 07.